# 11 YOLOv5s 结构复盘：每个模块怎样改变特征

这一课回应“把 YOLO 每层重新讲一遍”，只对照我们正在训练的 YOLOv5s 三类口罩模型。它与之前学习的 YOLOv1 结构不同。

以输入 $B\times3\times640\times640$ 为例：$B$ 是图片数量，3 是 RGB 通道数。

配置按编号 0–24 组织，共 25 个模块。一个模块内部可以有多层卷积，因此训练日志同时显示 125 layers 并不矛盾。我们先按模块理解，不逐个推导内部卷积参数。

## 1. 先看三部分分工

- Backbone：从图片提取不同层级的特征。
- Neck：让不同分辨率、不同层级的特征交流和融合。
- Detect Head：将融合特征变成框、目标存在性与类别的预测。

最终预测之后，还要解码、按分数筛选、进行 NMS，才能得到画在图片上的检测框。NMS 是后处理，不是这里的卷积模块。

按本地配置，0–9 属于骨干，10–23 主要完成多尺度融合，24 是 Detect。YAML 将后两部分一起放进 head 配置字段，所以配置里的字段名与讲解时的 Neck/Head 分工不完全一一对应。

## 2. 先认识六种积木

| 名字 | 做什么 | 怎样理解 |
|---|---|---|
| Conv | 卷积、BatchNorm、SiLU 激活 | 提取与变换特征；步长 2 时还会下采样 |
| C3 | 两个分支加工、拼接，再用卷积融合，内部含 Bottleneck | 在保留部分信息的同时进一步学习特征 |
| SPPF | 连续进行步长 1 的最大池化，并拼接不同阶段的结果 | 聚合不同范围的上下文，空间尺寸不变 |
| Upsample | 当前配置使用最近邻上采样 | 放大特征图的空间排列 |
| Concat | 沿通道维拼接 | 将两路信息放到一起，不是逐元素相加 |
| Detect | 当前模型有三个 1×1 输出卷积分支 | 在三个尺度上产生检测预测 |

C3 的两路都来自同一输入，一路经过较多加工、一路较短，再拼接融合；这不是简单把输入通道硬切成两半。内部 Bottleneck 在配置允许时可以使用残差相加。

Upsample 不会凭空恢复已经丢失的图像细节，因此还要与较高分辨率分支的特征融合。

## 3. 骨干：逐步缩小空间尺寸，提取更复杂的特征

下表的形状顺序固定为 $C\times H\times W$，省略前面的 batch 维。数字来自本轮训练实际打印的结构，不直接照抄 YAML 未缩放的通道数。

| 模块编号 | 类型 | 输出 C × H × W（省略 batch） | 作用 |
|---|---|---|---|
| 0 | Conv | $32\times320\times320$ | 6×6 卷积，步长 2，初步提取特征 |
| 1 | Conv | $64\times160\times160$ | 3×3 卷积，步长 2，下采样 |
| 2 | C3 | $64\times160\times160$ | C3，含 1 个 Bottleneck |
| 3 | Conv | $128\times80\times80$ | 3×3 卷积，步长 2，下采样 |
| 4 | C3 | $128\times80\times80$ | C3，含 2 个 Bottleneck；保留给后面融合 |
| 5 | Conv | $256\times40\times40$ | 3×3 卷积，步长 2，下采样 |
| 6 | C3 | $256\times40\times40$ | C3，含 3 个 Bottleneck；保留给后面融合 |
| 7 | Conv | $512\times20\times20$ | 3×3 卷积，步长 2，下采样 |
| 8 | C3 | $512\times20\times20$ | C3，含 1 个 Bottleneck |
| 9 | SPPF | $512\times20\times20$ | SPPF，聚合不同范围的上下文 |

骨干中较高分辨率的第 4 模块、较低分辨率的第 6 模块，以及深层第 9 模块的结果，会供后续多尺度融合使用。

空间尺寸变小不代表信息“完全消失”；网络在变换和压缩表示。较浅特征通常保留更细的空间信息，较深特征通常整合更大范围的信息。

## 4. Neck 前半段：把深层信息向高分辨率传递

| 模块编号 | 类型 | 输出 C × H × W（省略 batch） | 作用 |
|---|---|---|---|
| 10 | Conv | $256\times20\times20$ | 1×1 卷积，降低通道数 |
| 11 | Upsample | $256\times40\times40$ | 最近邻上采样，空间尺寸放大 2 倍 |
| 12 | Concat | $512\times40\times40$ | 与第 6 模块输出沿通道拼接 |
| 13 | C3 | $256\times40\times40$ | C3 加工融合后的特征 |
| 14 | Conv | $128\times40\times40$ | 1×1 卷积，降低通道数；保留给后面融合 |
| 15 | Upsample | $128\times80\times80$ | 最近邻上采样，空间尺寸放大 2 倍 |
| 16 | Concat | $256\times80\times80$ | 与第 4 模块输出沿通道拼接 |
| 17 | C3 | $128\times80\times80$ | C3，形成 P3 检测特征 |

抓住两次拼接即可：

- 第 12 模块把两路 $256\times40\times40$ 拼成 $512\times40\times40$。
- 第 16 模块把两路 $128\times80\times80$ 拼成 $256\times80\times80$。

拼接增加的是通道数量，空间尺寸必须一致。后面的 C3 再学习怎样融合这些信息。

第 17 模块输出 $128\times80\times80$，用于 P3 检测，通常更有利于小目标。它同时利用了深层传来的信息和较高分辨率的信息。

## 5. Neck 后半段：把融合后的信息再向低分辨率传递

| 模块编号 | 类型 | 输出 C × H × W（省略 batch） | 作用 |
|---|---|---|---|
| 18 | Conv | $128\times40\times40$ | 3×3 卷积，步长 2，下采样 |
| 19 | Concat | $256\times40\times40$ | 与第 14 模块输出沿通道拼接 |
| 20 | C3 | $256\times40\times40$ | C3，形成 P4 检测特征 |
| 21 | Conv | $256\times20\times20$ | 3×3 卷积，步长 2，下采样 |
| 22 | Concat | $512\times20\times20$ | 与第 10 模块输出沿通道拼接 |
| 23 | C3 | $512\times20\times20$ | C3，形成 P5 检测特征 |

前半段主要从 $20\times20$ 走到 $80\times80$，这一段再从 $80\times80$ 走向 $20\times20$。两条方向一起帮助不同尺度交换信息。

最终送往 Detect 的三路特征是：

| 来自模块 | 形状 | 相对于输入的步长 |
|---|---|---:|
| 17 | $B\times128\times80\times80$ | 8 |
| 20 | $B\times256\times40\times40$ | 16 |
| 23 | $B\times512\times20\times20$ | 32 |

小、中、大只是通常的检测倾向，不是目标尺寸的硬性分界。

## 6. 第 24 模块：特征图变成检测预测

三个尺度各自经过一个 1×1 输出卷积。在当前三类口罩任务中，每个 anchor 对应八个输出：四个框相关数值、一个目标存在性数值、三个类别数值。

每个位置有三个 anchor，所以输出卷积的通道数是：

$$
3\times(4+1+3)=24
$$

例如 P3：

$$
B\times128\times80\times80
\longrightarrow
B\times24\times80\times80
\longrightarrow
B\times3\times80\times80\times8
$$

三路原始输出为：

| 尺度 | 整理后的输出形状 |
|---|---|
| P3 | $B\times3\times80\times80\times8$ |
| P4 | $B\times3\times40\times40\times8$ |
| P5 | $B\times3\times20\times20\times8$ |

这些原始数值仍需激活和解码，四个框相关数值不能直接当成最终原图像素坐标。

推理时，解码后合并为每张图片 25200 条候选预测，每条八个数：

$$
3\times(80\times80+40\times40+20\times20)=25200
$$

它们是候选预测数量，不是真实人脸数量。后续通过类别分数筛选和 NMS 得到最终结果。

## 7. MLP 在哪里？微调又更新哪里？

当前 YOLOv5s 不需要先 Flatten，再通过一个 MLP 输出整张图的答案。检测头直接对各尺度特征做卷积，保留空间组织。

YOLOv1 使用全连接层产生最终预测，不能把它的实现直接套到 YOLOv5s 上。

本轮配置 freeze: [0] 表示不冻结层。Backbone、Neck、Detect 中的可训练参数都会更新。Upsample、Concat、最大池化本身没有可训练权重，但仍参与前向计算和信息传递。

所以“整个网络微调”是指允许各部分的可训练参数学习，不是每一种操作都有参数需要更新。

## 8. 当前阶段只需掌握这些问题

1. Backbone 输出的是最终类别，还是多层级特征图？
2. Concat 两路 $128\times80\times80$ 后，形状是什么？
3. Upsample 为什么还需要与较高分辨率的分支融合？
4. 当前三类模型，Detect 每个位置的输出卷积为什么是 24 个通道？
5. 微调整个网络，是否意味着 Concat 也有权重需要更新？

<details>
<summary>参考答案</summary>

1. 多层级特征图。
2. $256\times80\times80$，通道增加，空间尺寸不变。
3. 放大特征排列不会凭空恢复细节，融合可以引入另一层级的信息。
4. 三个 anchor，每个对应八个输出，合计 24。
5. 不是。Concat 本身没有可训练权重。

</details>

无需背住每一行编号。能沿“提取特征、融合特征、产生预测”解释图片怎样走过网络，就足以继续当前微调实践。

## 本课依据

- 本轮 runs/logs/train-expanded-30epochs.log 中打印的模型结构：25 个模块、125 layers、三类 Detect。
- vendor/yolov5/models/yolov5s.yaml：模块连接关系和尺度。
- vendor/yolov5/models/common.py：Conv、C3、Bottleneck、SPPF、Concat 的实际实现。
- vendor/yolov5/models/yolo.py：宽度/深度缩放与 Detect 输出整理。

表格形状已根据本轮日志中的实际输入通道、输出通道、卷积核、步长和连接关系进行独立的形状计算，未额外加载模型或占用训练显存。

本课没有更改正在运行的训练。